# TrakrAI GPU worker for Google Colab

This notebook lends a Colab GPU to **your factory's** model training in TrakrAI. It only runs jobs for the factory that created the pairing code, and it stops when you stop this notebook.

**Before you start**

1. This notebook asks Colab for a **T4 GPU** when it opens. If Colab shows a different runtime, choose **Runtime → Change runtime type → T4 GPU** and **Save**.
2. Use a Colab account with a paid plan or a positive compute-unit balance. Colab's terms do not allow running workers like this one on the free tier.
3. In TrakrAI, open **Model training → Colab GPU**, click **Create pairing code** and copy the code. A code works once and expires after 15 minutes.

**Run it**

Check that `TRAKRAI_URL` is your TrakrAI address, paste the code into `PAIRING_CODE` (or leave it empty and paste it when asked), then **Runtime → Run all**. The cell keeps running while the worker is connected; its log appears underneath.

**Stop it**

Press the stop button on the cell, or **Runtime → Disconnect and delete runtime**. A job that was running is handed back to TrakrAI and picked up again later. Your compute-unit usage is shown under **Runtime → View resources** and in Colab's billing page; TrakrAI shows which jobs this worker ran and for how long.


In [ ]:
# @title Connect this runtime to TrakrAI
TRAKRAI_URL = "https://ai.trakr.live"  # @param {type:"string"}
PAIRING_CODE = ""  # @param {type:"string"}
JOB_SLOTS = "train_detector:1,extract_frames:1,pretag_batch:1"  # @param {type:"string"}

import io
import json
import os
import shutil
import sys
import tarfile
import urllib.error
import urllib.request
from pathlib import Path

ROOT = Path("/content/trakrai")
PKG = ROOT / "pkg"
STATE = ROOT / "worker.json"
ROOT.mkdir(parents=True, exist_ok=True)

if shutil.which("nvidia-smi") is None:
    raise SystemExit(
        "No GPU on this runtime. Choose Runtime > Change runtime type > T4 GPU and run again."
    )


def post(path, body):
    headers = {"Content-Type": "application/json"}
    request = urllib.request.Request(
        TRAKRAI_URL + path, data=json.dumps(body).encode(), headers=headers, method="POST"
    )
    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            return json.loads(response.read() or b"{}")
    except urllib.error.HTTPError as error:
        detail = error.read().decode(errors="replace")
        try:
            detail = json.loads(detail).get("error", detail)
        except ValueError:
            pass
        raise SystemExit(f"TrakrAI said: {detail}") from None


token = json.loads(STATE.read_text())["token"] if STATE.exists() else None
if token is None:
    code = PAIRING_CODE.strip() or input("Pairing code from TrakrAI: ").strip()
    paired = post("/api/ml-worker/pair", {"code": code, "hostname": os.uname().nodename})
    token = paired["token"]
    STATE.write_text(json.dumps({"token": token, "name": paired["name"]}))
    STATE.chmod(0o600)
    print(f"Paired as {paired['name']} for {paired['factoryName']}.")

request = urllib.request.Request(
    TRAKRAI_URL + "/api/ml-worker/package", headers={"Authorization": "Bearer " + token}
)
try:
    with urllib.request.urlopen(request, timeout=300) as response:
        archive = tarfile.open(fileobj=io.BytesIO(response.read()), mode="r:gz")
except urllib.error.HTTPError:
    STATE.unlink(missing_ok=True)
    raise SystemExit(
        "This worker's token is no longer valid. Create a new pairing code and run again."
    ) from None
shutil.rmtree(PKG, ignore_errors=True)
try:
    archive.extractall(PKG, filter="data")
except TypeError:
    archive.extractall(PKG)
sys.path.insert(0, str(PKG))

from trakrai_ml_worker.colab import run_worker

exit_code = run_worker(TRAKRAI_URL, token, PKG, slots=JOB_SLOTS)
print(f"Worker stopped (exit code {exit_code}).")
if exit_code == 3:
    STATE.unlink(missing_ok=True)